# Aula 11 - Protocolo Agente-para-Agente (A2A)


## Configuração


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## O que é o Protocolo A2A?

O **protocolo Agent-to-Agent (A2A)** é um padrão aberto que permite que agentes de IA comuniquem-se,
descubram-se e colaborem — mesmo quando são construídos em diferentes frameworks ou hospedados
por diferentes serviços.

Conceitos chave:

- **Descoberta** – Os agentes publicam um *Cartão de Agente* que descreve as suas capacidades, facilitando
  que outros agentes (ou orquestradores) encontrem o especialista certo para uma tarefa.
- **Passagem de Mensagens** – Os agentes trocam mensagens estruturadas através de um protocolo comum, pelo que um
  pedido de um agente pode ser compreendido e realizado por outro independentemente da sua implementação
  interna.
- **Ciclo de Vida da Tarefa** – O A2A define estados como *submetida*, *a trabalhar*, *concluída* e
  *falhada*, dando ao orquestrador total visibilidade de como uma tarefa delegada está a progredir.

Nesta lição simulamos a colaboração ao estilo A2A ligando três agentes de viagem especializados
num fluxo de trabalho onde cada agente contribui com a sua experiência e passa os resultados ao seguinte.


## Criar Agentes de Viagens Especializados


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Colaboração Multi-Agente através de Workflow

Ligamos os três agentes num workflow sequencial que espelha a passagem de mensagens A2A:

1. **CurrencyExchangeAgent** recebe o pedido do utilizador e produz orientações sobre moeda.
2. **ActivityPlannerAgent** recebe o contexto enriquecido e acrescenta recomendações de atividades.
3. **TravelManagerAgent** sintetiza ambas as entradas num resumo final de viagem.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Compreender A2A em Produção

Num ambiente de produção, o protocolo A2A desbloqueia cenários poderosos entre serviços:

| Capacidade | Descrição |
|---|---|
| **Interoperabilidade entre frameworks** | Um agente construído com um framework pode delegar tarefas a um agente construído com qualquer outro framework compatível com A2A, permitindo verdadeira interoperabilidade entre organizações. |
| **Fronteiras de serviço** | Os agentes podem viver em microsserviços separados, regiões na nuvem ou até mesmo em organizações diferentes, enquanto colaboram sem interrupções. |
| **Descoberta dinâmica** | Um orquestrador pode consultar um registo de Agent Cards em tempo de execução para encontrar o especialista mais adequado para uma sub-tarefa específica. |
| **Transmissão e notificações push** | O A2A suporta Server-Sent Events (SSE) para atualizações em tempo real do progresso e notificações push para tarefas de longa duração. |

O fluxo de trabalho que construímos acima é uma versão simplificada, em processo, deste padrão. Numa implantação real,
cada agente exporia um endpoint HTTP, publicaria um Agent Card e comunicaria
via o protocolo A2A JSON-RPC.


## Resumo

Nesta lição, aprendeu:

1. **O que é o protocolo A2A** — um padrão aberto para descoberta, comunicação,
   e gestão de tarefas entre agentes.
2. **Como criar agentes especializados** — um agente de Câmbio de Moeda, um agente
   de Planeamento de Atividades, e um orquestrador Gestor de Viagens.
3. **Como ligar agentes num fluxo de trabalho** — usando `WorkflowBuilder` para modelar a passagem sequencial
   de mensagens entre agentes.
4. **Como o A2A funciona em produção** — permitindo colaboração entre frameworks e serviços
   com descoberta dinâmica e atualizações em streaming.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
